In [80]:
import numpy as np 
import pandas as pd

In [81]:
match = pd.read_csv("matches.csv")

In [82]:
delivery = pd.read_csv("deliveries.csv")

In [83]:
delivery.head(1)

,match_id,inning,batting_team,bowling_team,over,ball,batsman,non_striker,bowler,is_super_over,...,bye_runs,legbye_runs,noball_runs,penalty_runs,batsman_runs,extra_runs,total_runs,player_dismissed,dismissal_kind,fielder
0,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,1,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN


In [84]:
total_score_df = delivery.groupby(['match_id','inning']).sum()['total_runs'].reset_index()

In [85]:
total_score_df = total_score_df[total_score_df['inning'] == 1]

In [86]:
match_df = match.merge(total_score_df[['match_id','total_runs']],left_on = 'id', right_on = 'match_id')

In [87]:
match_df['team1'].unique()

array(['Sunrisers Hyderabad', 'Mumbai Indians', 'Gujarat Lions',
       'Rising Pune Supergiant', 'Royal Challengers Bangalore',
       'Kolkata Knight Riders', 'Delhi Daredevils', 'Kings XI Punjab',
       'Chennai Super Kings', 'Rajasthan Royals', 'Deccan Chargers',
       'Kochi Tuskers Kerala', 'Pune Warriors', 'Rising Pune Supergiants',
       'Delhi Capitals'], dtype=object)

In [88]:
teams = [
    'Sunrisers Hyderabad',
    'Mumbai Indians',
    'Royal Challengers Bangalore',
    'Kolkata Knight Riders',
    'Kings XI Punjab',
    'Chennai Super Kings',
    'Rajasthan Royals',
    'Delhi Capitals'
]

In [89]:
match_df['team1'] = match_df['team1'].str.replace('Delhi Daredevils','Delhi Capitals')
match_df['team2'] = match_df['team2'].str.replace('Delhi Daredevils','Delhi Capitals')

match_df['team1'] = match_df['team1'].str.replace('Deccan Chargers','Sunrisers Hyderabad')
match_df['team2'] = match_df['team2'].str.replace('Deccan Chargers','Sunrisers Hyderabad')

In [90]:
match_df = match_df[match_df['team1'].isin(teams)]
match_df = match_df[match_df['team2'].isin(teams)]

In [91]:
match_df = match_df[match_df['dl_applied'] == 0]

In [92]:
match_df = match_df[['match_id','city','winner','total_runs']]

In [93]:
delivery_df = match_df.merge(delivery, on = 'match_id')

In [94]:
delivery_df = delivery_df[delivery_df['inning'] == 2]

In [95]:
delivery_df['current_score'] = delivery_df.groupby('match_id')['total_runs_y'].cumsum()

In [96]:
delivery_df['runs_left'] = delivery_df['total_runs_x'] - delivery_df['current_score']

In [97]:
delivery_df['balls_left'] = 126 - (delivery_df['over']*6 + delivery_df['ball'])

In [99]:
delivery_df['player_dismissed'] = delivery_df['player_dismissed'].fillna("0")

In [100]:
delivery_df['player_dismissed'] = delivery_df['player_dismissed'].apply(lambda x:x if x == "0" else "1")
delivery_df['player_dismissed'] = delivery_df['player_dismissed'].astype('int')
wickets = delivery_df.groupby('match_id')['player_dismissed'].cumsum().values
delivery_df['wickets_left'] = 10 - wickets

In [101]:
delivery_df.head()

,match_id,city,winner,total_runs_x,inning,batting_team,bowling_team,over,ball,batsman,...,batsman_runs,extra_runs,total_runs_y,player_dismissed,dismissal_kind,fielder,current_score,runs_left,balls_left,wickets_left
125,1,Hyderabad,Sunrisers Hyderabad,207,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,1,CH Gayle,...,1,0,1,0,NaN,NaN,1,206,119,10
126,1,Hyderabad,Sunrisers Hyderabad,207,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,2,Mandeep Singh,...,0,0,0,0,NaN,NaN,1,206,118,10
127,1,Hyderabad,Sunrisers Hyderabad,207,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,3,Mandeep Singh,...,0,0,0,0,NaN,NaN,1,206,117,10
128,1,Hyderabad,Sunrisers Hyderabad,207,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,4,Mandeep Singh,...,2,0,2,0,NaN,NaN,3,204,116,10
129,1,Hyderabad,Sunrisers Hyderabad,207,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,5,Mandeep Singh,...,4,0,4,0,NaN,NaN,7,200,115,10


In [103]:
delivery_df['crr'] = (delivery_df['current_score']*6)/(120 - delivery_df['balls_left'])

In [104]:
delivery_df['rrr'] = (delivery_df['runs_left']*6)/delivery_df['balls_left']

In [106]:
def result(row):
    return 1 if row['batting_team'] == row['winner'] else 0

In [107]:
delivery_df['result'] = delivery_df.apply(result, axis = 1)

In [111]:
final_df = delivery_df[['batting_team','bowling_team','city','runs_left','balls_left','wickets_left','total_runs_x','crr','rrr','result']]

In [114]:
final_df = final_df.sample(final_df.shape[0])

In [130]:
final_df.dropna(inplace = True)

In [139]:
final_df = final_df[final_df['balls_left'] != 0]

In [140]:
from sklearn.model_selection import train_test_split

In [141]:
X = final_df.iloc[:,:-1]

In [142]:
y = final_df.iloc[:,-1]

In [143]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2, random_state = 1)

In [144]:
X_train

,batting_team,bowling_team,city,runs_left,balls_left,wickets_left,total_runs_x,crr,rrr
96375,Royal Challengers Bangalore,Chennai Super Kings,Ranchi,99,66,9,138,4.333333,9.000000
48645,Deccan Chargers,Kolkata Knight Riders,Kolkata,162,118,10,163,3.000000,8.237288
87525,Rajasthan Royals,Sunrisers Hyderabad,Abu Dhabi,26,26,7,133,6.829787,6.000000
376,Delhi Daredevils,Royal Challengers Bangalore,Bangalore,155,115,10,157,2.400000,8.086957
76837,Chennai Super Kings,Kolkata Knight Riders,Kolkata,82,73,8,119,4.723404,6.739726
...,...,...,...,...,...,...,...,...,...
137835,Royal Challengers Bangalore,Sunrisers Hyderabad,Hyderabad,211,99,8,233,6.285714,12.787879
112644,Kolkata Knight Riders,Rajasthan Royals,Mumbai,178,100,8,199,6.300000,10.680000
138092,Rajasthan Royals,Chennai Super Kings,Chennai,155,92,7,182,5.785714,10.108696
142921,Rajasthan Royals,Kings XI Punjab,Mohali,162,103,10,190,9.882353,9.436893


In [145]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

trf = ColumnTransformer([
    ('trf',OneHotEncoder(sparse_output=False,drop='first'),['batting_team','bowling_team','city'])
]
,remainder='passthrough')

In [161]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

In [162]:
pipe = Pipeline(steps=[
    ('step1',trf),
    ('step2',LogisticRegression(solver='liblinear'))
])

In [163]:
pipe.fit(X_train,y_train)

,steps,"[('step1', ...), ('step2', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('trf', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [164]:
y_pred = pipe.predict(X_test)

In [165]:
from sklearn.metrics import accuracy_score

In [166]:
accuracy_score(y_test,y_pred)

0.8029995094260285

In [167]:
teams

['Sunrisers Hyderabad',
 'Mumbai Indians',
 'Royal Challengers Bangalore',
 'Kolkata Knight Riders',
 'Kings XI Punjab',
 'Chennai Super Kings',
 'Rajasthan Royals',
 'Delhi Capitals']

In [168]:
delivery_df['city'].unique()

array(['Hyderabad', 'Bangalore', 'Mumbai', 'Indore', 'Kolkata', 'Delhi',
       'Chandigarh', 'Jaipur', 'Chennai', 'Cape Town', 'Port Elizabeth',
       'Durban', 'Centurion', 'East London', 'Johannesburg', 'Kimberley',
       'Bloemfontein', 'Ahmedabad', 'Cuttack', 'Nagpur', 'Dharamsala',
       'Visakhapatnam', 'Pune', 'Raipur', 'Ranchi', 'Abu Dhabi',
       'Sharjah', nan, 'Mohali', 'Bengaluru'], dtype=object)

In [169]:
import pickle
pickle.dump(pipe,open('pipe.pkl','wb'))